In [1]:
!git clone https://github.com/deepanrajm/GL_MML.git

Cloning into 'GL_MML'...
remote: Enumerating objects: 540, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 540 (delta 0), reused 1 (delta 0), pack-reused 536 (from 1)
Receiving objects: 100% (540/540), 309.77 MiB | 14.57 MiB/s, done.
Resolving deltas: 100% (93/93), done.
Updating files: 100% (500/500), done.


In [ ]:
# Install
!pip install hmmlearn librosa opencv-python scikit-learn

In [ ]:
import os
import numpy as np
import cv2
import librosa
from hmmlearn import hmm
from sklearn.decomposition import PCA
from collections import Counter

In [ ]:

FACE_DIR = "GL_MML/Unit_4/face_images"
VOICE_DIR = "GL_MML/Unit_4/voices_for_images"

# -----------------------------
# LOAD + MATCH FILES
# -----------------------------
def load_data(face_dir, voice_dir):
    face_files = sorted(os.listdir(face_dir))
    voice_files = sorted(os.listdir(voice_dir))

    data = []
    labels = []

    for f in face_files:
        name = f.split('.')[0]  # dhoni_1
        voice_file = name + ".wav"

        if voice_file in voice_files:
            face_path = os.path.join(face_dir, f)
            voice_path = os.path.join(voice_dir, voice_file)

            label = name.split('_')[0]  # dhoni

            data.append((face_path, voice_path))
            labels.append(label)

    return data, labels

data, labels = load_data(FACE_DIR, VOICE_DIR)

print("Samples:", len(data))
print("Labels:", set(labels))

In [ ]:

# -----------------------------
# FEATURE EXTRACTION
# -----------------------------
def extract_face(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (64, 64))
    return img.flatten()

def extract_voice(path):
    y, sr = librosa.load(path, sr=None)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    return np.mean(mfcc, axis=1)

face_feat = []
voice_feat = []

for fpath, vpath in data:
    face_feat.append(extract_face(fpath))
    voice_feat.append(extract_voice(vpath))

face_feat = np.array(face_feat)
voice_feat = np.array(voice_feat)

# -----------------------------
# REDUCE DIM
# -----------------------------
pca = PCA(n_components=5) # Changed n_components from 30 to 5
face_feat = pca.fit_transform(face_feat)

# -----------------------------
# COMBINE (Factorial)
# -----------------------------
X = np.hstack([face_feat, voice_feat])

In [ ]:
# -----------------------------
# TRAIN HMM
# -----------------------------
model = hmm.GaussianHMM(n_components=3, covariance_type="full", n_iter=100)
model.fit(X)

hidden_states = model.predict(X)

print("\nHidden states:", hidden_states)

# -----------------------------
# ANALYZE CLUSTERS
# -----------------------------
for i in range(3):
    cluster_labels = [labels[j] for j in range(len(labels)) if hidden_states[j] == i]
    print(f"\nCluster {i}:")
    print(Counter(cluster_labels))